# IBM — Python Project for Data Science
## Analyzing Historical Stock/Revenue Data and Building a Dashboard

Seis exercícios: preços TSLA, receita Tesla, preços GME, receita GameStop e dois painéis. Execute **Run All** no ambiente Coursera (Internet necessária) e salve o notebook com as saídas antes da avaliação. Fontes: `yfinance` e páginas de receitas arquivadas fornecidas pela IBM.

In [ ]:
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

def scrape_quarterly_revenue(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    tables = soup.find_all("table")
    if len(tables) < 2:
        raise ValueError("A tabela trimestral não foi encontrada: " + url)
    quarterly = next((table for table in tables if "Quarterly Revenue" in table.get_text(" ", strip=True)), tables[1])
    rows = []
    for tr in quarterly.find_all("tr"):
        cols = tr.find_all("td")
        if len(cols) >= 2:
            rows.append((cols[0].get_text(strip=True), cols[1].get_text(strip=True)))
    df = pd.DataFrame(rows, columns=["Date", "Revenue"])
    df["Revenue"] = df["Revenue"].str.replace(r"[$,]", "", regex=True)
    df = df[df["Revenue"].str.strip().ne("")].reset_index(drop=True)
    return df

def make_graph(stock_data, revenue_data, title, end_date="2021-06-30"):
    prices = stock_data.copy()
    revenues = revenue_data.copy()
    prices["Date"] = pd.to_datetime(prices["Date"], utc=True).dt.tz_localize(None)
    revenues["Date"] = pd.to_datetime(revenues["Date"], errors="coerce")
    revenues["Revenue"] = pd.to_numeric(revenues["Revenue"], errors="coerce")
    prices = prices[prices["Date"] <= end_date].sort_values("Date")
    revenues = revenues[revenues["Date"] <= end_date].dropna().sort_values("Date")
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    ax1.plot(prices["Date"], prices["Close"], color="royalblue", linewidth=1.5)
    ax2.plot(revenues["Date"], revenues["Revenue"], color="seagreen", linewidth=1.7)
    ax1.set_title(f"{title} - Historical Share Price")
    ax2.set_title(f"{title} - Historical Revenue")
    ax1.set_ylabel("Price (USD)")
    ax2.set_ylabel("Revenue (USD Millions)")
    ax2.set_xlabel("Date")
    ax2.xaxis.set_major_locator(mdates.YearLocator(2))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.tight_layout()
    plt.show()
    return fig

## Question 1 — Use yfinance to Extract Stock Data (Tesla)

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
display(tesla_data.head())

## Question 2 — Use Webscraping to Extract Tesla Revenue Data

In [ ]:
tesla_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = scrape_quarterly_revenue(tesla_url)
display(tesla_revenue.tail())

## Question 3 — Use yfinance to Extract Stock Data (GameStop)

In [ ]:
gamestop = yf.Ticker("GME")
gme_data = gamestop.history(period="max")
gme_data.reset_index(inplace=True)
display(gme_data.head())

## Question 4 — Use Webscraping to Extract GME Revenue Data

In [ ]:
gamestop_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
gme_revenue = scrape_quarterly_revenue(gamestop_url)
display(gme_revenue.tail())

## Question 5 — Plot Tesla Stock Graph

In [ ]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6 — Plot GameStop Stock Graph

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop")

## Final checks

Capture screenshots from each question heading to its code and displayed output. Run **Kernel → Restart & Run All** in the Coursera notebook before saving for submission. This GitHub version contains executable source code; its online outputs are generated on execution. The stock-data source may change over time because yfinance provides current adjusted historical records.